In [9]:
from sklearn.impute import SimpleImputer
import pandas as pd
from typing import Literal
import importlib
from sklearn.metrics import mean_squared_error
from sklearn.base import clone
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..','..')))


from API.Requests import projectRequests
from sklearn import model_selection
import joblib

In [10]:
state={
   'X_columns': ['Embarked', 'Sex', 'Age', 'Parch', 'Fare', 'SibSp', 'Pclass'],
'y_column': 'Survived',
'test_size': 0.2,
    'shuffle': True, 
    'cross_validation': False,
    'val_size': 0.2, 
    'stratify': True,
    'val_size': 0.2,
    'y_column': 'Survived',
}
project_id='67c1ba76e833b024ca9cb615'

In [11]:
df= await projectRequests.get_dataset(project_id)
X=df[state['X_columns']]
y=df[state['y_column']]

X_train,X_test, y_train, y_test=model_selection.train_test_split(X,y,test_size=state['test_size'],shuffle=state['shuffle'],stratify=y if state['stratify'] else None,random_state=42)

if state["cross_validation"]:
        if state['stratify']:
            kf=model_selection.StratifiedKFold(n_splits=state['n_splits'], shuffle=state['shuffle'], random_state=42)
        else:
            kf=model_selection.KFold(n_splits=state['n_splits'], shuffle=state['shuffle'], random_state=42)

else:
        X_train, X_val, y_train, y_val = model_selection.train_test_split(X_train, y_train, test_size=state['val_size'], shuffle=state['shuffle'], stratify=y_train if state['stratify'] else None, random_state=42)
X_train['row_id'] = range(len(X_train))
y_train = pd.DataFrame({state['y_column']: y_train, 'row_id': range(len(y_train))})
X_val['row_id'] = range(len(X_val))
y_val = pd.DataFrame({state['y_column']: y_val, 'row_id': range(len(y_val))})

In [12]:
def get_X_pipeline(project_id):
    pipeline_path = r"D:\UNI\Graduation Project\PROJECT\C.A.S.E-Automated-Data-Analysis-By-LLMs\static"+f"/{project_id}_X_pipeline.pkl"
    if os.path.exists(pipeline_path):
        return joblib.load(pipeline_path)
    else:
        return None
    
def get_Y_pipeline(project_id):
    pipeline_path = r"D:\UNI\Graduation Project\PROJECT\C.A.S.E-Automated-Data-Analysis-By-LLMs\static"+f"/{project_id}_Y_pipeline.pkl"
    if os.path.exists(pipeline_path):
        return joblib.load(pipeline_path)
    else:
        return None

In [13]:
Xpreprocessing_pipeline=get_X_pipeline(project_id)
Ypreprocessing_pipeline=get_Y_pipeline(project_id)
Xpreprocessing_pipeline

C:\Users\Kareem Abouelseoud\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.5.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Kareem Abouelseoud\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.6.1 when using version 1.5.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.wa

ColumnTransformer(remainder='passthrough', sparse_threshold=0,
                  transformers=[('Preprocess_Age',
                                 Pipeline(steps=[('Null Handler Age',
                                                  NullValueTransformer(feature_name='Age',
                                                                       strategy='fill_mean')),
                                                 ('Scaler', StandardScaler()),
                                                 ('Outlier Handler',
                                                  OutlierTransformer(feature_name='Age',
                                                                     strategy='winsorize'))]),
                                 ['Age']),
                                ('Preprocess_Embarked',
                                 Pipeline(steps=[('N...
                                                  OutlierTransformer(feature_name='Fare',
                                                                     strategy='winsorize'))]),
                                 ['Fare']),
                                ('Preprocess_SibSp',
                                 Pipeline(steps=[('Scaler', MinMaxScaler()),
                                                 ('Outlier Handler',
                                                  OutlierTransformer(feature_name='SibSp',
                                                                     strategy='winsorize'))]),
                                 ['SibSp']),
                                ('Preprocess_Parch',
                                 Pipeline(steps=[('Scaler', MinMaxScaler()),
                                                 ('Outlier Handler',
                                                  OutlierTransformer(feature_name='Parch',
                                                                     strategy='winsorize'))]),
                                 ['Parch'])])

In [14]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,None,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,None,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,None,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,None,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,None,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [15]:
from sklearn.compose import ColumnTransformer
def safe_column_transformer(transformers, df):
    valid_transformers = []
    
    for name, transformer, cols in transformers:
        try:
            # Try fitting on a small sample
            _ = transformer.fit_transform(df[cols])
            valid_transformers.append((name, transformer, cols))
        except Exception as e:
            print(f"⚠️ Removing '{name}' due to error: {e}")  # Log the issue

    return ColumnTransformer(valid_transformers, remainder='passthrough')



In [16]:
if Xpreprocessing_pipeline:
    numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
    categorical_cols = X_train.select_dtypes(include=['object']).columns
    Xpreprocessing_pipeline.transformers = [t for t in Xpreprocessing_pipeline.transformers if t is not None]
        
    # Remove duplicates from transformers
    seen_transformers = set()
    unique_transformers = []
    for transformer in Xpreprocessing_pipeline.transformers:
        if not transformer[1].steps:
                continue
        if transformer[0] not in seen_transformers:
            unique_transformers.append(transformer)
            seen_transformers.add(transformer[0])
    Xpreprocessing_pipeline.transformers = unique_transformers
    
    if Xpreprocessing_pipeline.transformers[0][0]=='Drop':
        X_Dropper=Xpreprocessing_pipeline.transformers.pop(0)
        X_temp=X_Dropper[1].fit_transform(X_train)
    
    else:
        X_Dropper=None
        X_temp=X_train
    Xpreprocessing_pipeline = safe_column_transformer(Xpreprocessing_pipeline.transformers, X_temp)
    if Xpreprocessing_pipeline.transformers:
        X_temp=Xpreprocessing_pipeline.fit_transform(X_temp)
        columns=Xpreprocessing_pipeline.get_feature_names_out()
        columns=[column.split('__',1)[1] if '__' in column else column for column in columns]
        X_temp=pd.DataFrame(X_temp,columns=columns)
    
    final_imputer=SimpleImputer(strategy='median')
    X_temp=final_imputer.fit_transform(X_temp)
    X_temp=pd.DataFrame(X_temp,columns=columns)
else:
    X_temp=X_train

if Ypreprocessing_pipeline:
    Ypreprocessing_pipeline.transformers = [t for t in Ypreprocessing_pipeline.transformers if t is not None]
    
    seen_transformers = set()
    unique_transformers = []
    for transformer in Ypreprocessing_pipeline.transformers:
        if not transformer[1].steps:
            continue
        if transformer[0] not in seen_transformers:
            unique_transformers.append(transformer)
            seen_transformers.add(transformer[0])
    
    Ypreprocessing_pipeline.transformers = unique_transformers
    
    
    if Ypreprocessing_pipeline.transformers[0][0]=='Drop':
        Y_Dropper=Ypreprocessing_pipeline.transformers.pop(0)
        y_temp=Y_Dropper[1].fit_transform(y_train)
    else:
        Y_Dropper=None
        y_temp=y_train
    
    if Ypreprocessing_pipeline.transformers:
        y_temp=Ypreprocessing_pipeline.fit_transform(y_temp)
        columns=Ypreprocessing_pipeline.get_feature_names_out()
        columns=[column.split('__',1)[1] if '__' in column else column for column in columns]
        y_temp=pd.DataFrame(y_temp,columns=columns)

else:
    y_temp=y_train

merged=X_temp.merge(y_temp, on='row_id',how='inner')
X_train = merged.drop(columns=['row_id', state['y_column']])
y_train = merged[state['y_column']]


if Xpreprocessing_pipeline:
    if X_Dropper:
        X_val_temp=X_Dropper[1].transform(X_val)
    else:
        X_val_temp=X_val

    if Xpreprocessing_pipeline.transformers:
        X_val_temp=Xpreprocessing_pipeline.transform(X_val_temp)
        columns=Xpreprocessing_pipeline.get_feature_names_out()
        columns=[column.split('__',1)[1] if '__' in column else column for column in columns]
        X_val_temp=pd.DataFrame(X_val_temp,columns=columns)
    
    X_val_temp=final_imputer.transform(X_val_temp)
    X_val_temp=pd.DataFrame(X_val_temp,columns=columns)
else:
    X_val_temp=X_val
    
if Ypreprocessing_pipeline:
    if Y_Dropper:
        y_val_temp=Y_Dropper[1].transform(y_val)
    else:
        y_val_temp=y_val
    if Ypreprocessing_pipeline.transformers:
        y_val_temp=Ypreprocessing_pipeline.transform(y_val)
        columns=Ypreprocessing_pipeline.get_feature_names_out()
        columns=[column.split('__',1)[1] if '__' in column else column for column in columns]
        y_val_temp=pd.DataFrame(y_val_temp,columns=columns)
else:
    y_val_temp=y_val

merged=X_val_temp.merge(y_val_temp, on='row_id',how='inner')
X_val = merged.drop(columns=['row_id', state['y_column']])
y_val = merged[state['y_column']]

print("Removing object data types from X_train and X_val")
print(f"Object columns in X_train: {X_train.select_dtypes(include=['object']).columns.tolist()}")
print(f"Object columns in X_val: {X_val.select_dtypes(include=['object']).columns.tolist()}")

X_train = X_train.select_dtypes(exclude=['object'])
X_val = X_val.select_dtypes(exclude=['object'])

# Ensure both X_train and X_val contain the same features
common_columns = X_train.columns.intersection(X_val.columns)
X_train = X_train[common_columns]
X_val = X_val[common_columns]

y_train = y_train.dropna()
y_val = y_val.dropna()



Removing object data types from X_train and X_val
Object columns in X_train: []
Object columns in X_val: []


In [17]:
X_test['row_id'] = range(len(X_test))
y_test = pd.DataFrame({state['y_column']: y_test, 'row_id': range(len(y_test))})
if Xpreprocessing_pipeline:
    if X_Dropper:
        X_test_temp=X_Dropper[1].transform(X_test)
    else:
        X_test_temp=X_test

    if Xpreprocessing_pipeline.transformers:
        X_test_temp=Xpreprocessing_pipeline.transform(X_test_temp)
        columns=Xpreprocessing_pipeline.get_feature_names_out()
        columns=[column.split('__',1)[1] if '__' in column else column for column in columns]
        X_test_temp=pd.DataFrame(X_test_temp,columns=columns)

    X_test_temp=final_imputer.transform(X_test_temp)
    X_test_temp=pd.DataFrame(X_test_temp,columns=columns)

if Ypreprocessing_pipeline:
    if Y_Dropper:
        y_test_temp=Y_Dropper[1].transform(y_test)
    else:
        y_test_temp=y_test

    if Ypreprocessing_pipeline.transformers:
        y_test_temp=Ypreprocessing_pipeline.transform(y_test_temp)
        columns=Ypreprocessing_pipeline.get_feature_names_out()
        columns=[column.split('__',1)[1] if '__' in column else column for column in columns]
        y_test_temp=pd.DataFrame(y_test_temp,columns=columns)

    merged=X_test_temp.merge(y_test_temp, on='row_id',how='inner')
    X_test = merged.drop(columns=['row_id', state['y_column']])
    y_test = merged[state['y_column']]


In [18]:
def get_model(project_id, model_name):
    model_path = rf"D:\UNI\Graduation Project\PROJECT\C.A.S.E-Automated-Data-Analysis-By-LLMs\static\{project_id}_{model_name}_model.pkl"
    if os.path.exists(model_path):
        return joblib.load(model_path)
    else:
        return None

In [ ]:
model=get_model(project_id, 'Light Gradient Boosting Machine (LightGBM) Classifier')

C:\Users\Kareem Abouelseoud\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DummyClassifier from version 1.6.1 when using version 1.5.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Kareem Abouelseoud\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.6.1 when using version 1.5.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations

In [24]:
X_train.columns

Index(['Age', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Sex_female',
       'Sex_male', 'Pclass_1', 'Pclass_2', 'Pclass_3', 'Fare', 'SibSp',
       'Parch'],
      dtype='object')

In [21]:
X_test.columns

Index(['Age', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Sex_female',
       'Sex_male', 'Pclass_1', 'Pclass_2', 'Pclass_3', 'Fare', 'SibSp',
       'Parch'],
      dtype='object')

In [25]:
model.score(X_test, y_test)

0.7988826815642458